# CD CosMx Resection: Raw Data Processing (12 samples)

QC, normalization, and tissue section DBSCAN assignment for all 12 resection samples.
See 06_cd_resection_anno.R for AUCell annotation.

In [ ]:
# ── Paths ──
DATA_DIR   = "/share/fsmresfiles/UC/AtoMx/CD_surgical_resection"
OUTPUT_DIR = DATA_DIR

# All 12 resection samples (generalized loop)
SAMPLES = [
    'Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_760',
    'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',
    'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',
    'Cos1_6KDC_strictCD_R1_B4_0715_11_08_2025_14_20_23_133',
    'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',
    'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',
    'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',
    'Cos3_6KDC_strictCD_R1_B5_0716_11_08_2025_14_17_56_277',
    'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
    'Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738',
    'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',
    'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952',
]

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import anndata as ad
from matplotlib.backends.backend_pdf import PdfPages

# reading in data and normalization

In [ ]:
import sys

In [ ]:
samples = [
'Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_760',                
'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',          
'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',       
'Cos1_6KDC_strictCD_R1_B4_0715_11_08_2025_14_20_23_133',     
'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',  
'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',  
'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',                  
'Cos3_6KDC_strictCD_R1_B5_0716_11_08_2025_14_17_56_277',                
'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
'Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738',
'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',         
'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952'  
]

In [ ]:
prefix = '/share/fsmresfiles/UC/AtoMx/CD_surgical_resection/'

In [ ]:
# prefix = directory that contains step1 reformatted sample folders e.g., '/share/fsmresfiles/UC/AtoMx/UST/cosmx/processed/'
# Renamed all sample folders to just sample name and response to keep things clean e.g., 'AGS1732860_AGS1918487_USTNR'

for i in samples:
    sample_dir_write = prefix+i+'/Processed/'
    sample_dir = prefix+i+'/dir_formatted'
    fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
    fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
    if 'FOV' in fov_df.columns:
      print("Refactoring file to older format.")
      # Rename 'FOV' column to 'fov'
      fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
      # have fov_file reference the new, formatted file and write it
      fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
      fov_df.to_csv(fov_file, index=False)
    
    for i in os.listdir(sample_dir):
        if 'exprMat' in i:
            sample = i.split('_exprMat_file.csv.gz')[0]
    print(sample)

    adata = sq.read.nanostring(
        path=sample_dir,
        counts_file=sample+"_exprMat_file.csv.gz",
        meta_file=sample+"_metadata_file.csv.gz",
        fov_file="fov_positions_formatted.csv")
    print ('adata read in')

    to_drop = []
    for i in adata.obs.columns:
        if 'clust' in i:
            to_drop.append(i)
    print (to_drop)

    adata.obs = adata.obs.drop(to_drop,axis = 1)
    os.makedirs(sample_dir_write, exist_ok=True)
    adata.write_h5ad(sample_dir_write+'raw_data.h5ad')
    print('saved')

In [ ]:
adata

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_spatial_fov(adata, sample_name, category_column="fov", save_pdf=None, 
                     figsize=(9, 10), point_size=1, legend_ncols=3):
    """
    Plot spatial coordinates colored by FOV categories with numeric ordering.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object containing spatial coordinates and observations
    sample_name : str
        Name of the sample for the plot title
    category_column : str, optional
        Column name in adata.obs containing FOV categories (default: "fov")
    save_pdf : str or None, optional
        If provided, save the plot to this PDF filepath (default: None)
    figsize : tuple, optional
        Figure size as (width, height) (default: (9, 10))
    point_size : float, optional
        Size of scatter plot points (default: 1)
    legend_ncols : int, optional
        Number of columns in the legend (default: 3)
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        The generated figure object
    """
    # Extract spatial coordinates
    coords = adata.obsm["spatial_fov"]
    fov_series = adata.obs[category_column].astype(str)
    
    # Build a numeric sort key from any digits in the label (e.g., "FOV_12" -> 12)
    num_key = pd.to_numeric(fov_series.str.extract(r'(\d+)')[0], errors="coerce")
    
    # Unique labels + numeric keys -> sort by number, then by label; non-numeric go last
    uniq = pd.DataFrame({"label": fov_series.unique()})
    uniq["num"] = pd.to_numeric(uniq["label"].str.extract(r'(\d+)')[0], errors="coerce")
    uniq = uniq.sort_values(["num", "label"], na_position="last")
    ordered_labels = uniq["label"].tolist()
    
    # Categorical with ordered categories
    cats = pd.Categorical(fov_series, categories=ordered_labels, ordered=True)
    codes = cats.codes
    
    # Palette and colors (tab20 cycles; expand if >20 categories)
    palette = sns.color_palette("tab20", len(ordered_labels))
    point_colors = [palette[c] if c >= 0 else (0.7, 0.7, 0.7) for c in codes]
    
    # Create plot
    fig = plt.figure(figsize=figsize)
    plt.scatter(coords[:, 0], coords[:, 1], c=point_colors, s=point_size)
    plt.title(f"{sample_name} Spatial Plot for FOVs")
    plt.xlabel("X coordinate")
    plt.ylabel("Y coordinate")
    
    # Legend in numeric order
    handles = [
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=palette[i], markersize=8, linestyle="None")
        for i in range(len(ordered_labels))
    ]
    
    plt.legend(
        handles, ordered_labels,
        title=category_column,
        loc="upper left",
        ncol=legend_ncols,
        frameon=False,
        columnspacing=1.0,
        handletextpad=0.4,
        prop={"size": 8},
        title_fontsize=9,
        bbox_to_anchor=(1.02, 1),
    )
    
    plt.tight_layout()
    
    # Save to PDF if requested
    if save_pdf:
        plt.savefig(save_pdf, format='pdf', bbox_inches='tight', dpi=300)
        print(f"Plot saved to: {save_pdf}")
    
    plt.close(fig)  # Close the figure to free memory
    
    return fig

# Example usage:
# plot_spatial_fov(adata, "Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_7", save_pdf="output.pdf")

In [ ]:
# save raw counts and normalized counts into csvs (separate by treatment), add metadata and save to h5ad
for sample in samples:
    print(sample)
    sample_dir_write = prefix+sample+'/Processed/'
    adata = sc.read_h5ad(sample_dir_write+'raw_data.h5ad')
    print('adata read in')

    # add in metadata
    adata.obs['sample'] = sample

    # remove negative probes and control
    adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
    adata.var['control_probe']=[True if 'SystemControl' in i else False for i in adata.var.index.tolist()]
    adata=adata[:, ~adata.var["neg_probe"]]
    adata=adata[:, ~adata.var["control_probe"]]

    # qc
    sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=True)
    sc.pl.violin(
            adata,
            ["n_genes_by_counts", "total_counts",'Area'],
            jitter=0.4,
            multi_panel=True,show=False)
    plt.savefig(sample_dir_write+'pre_violin_plot.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.close()


In [ ]:
# qc thresholds for filtering
ngenes_top= 2500
ngenes_bot= 50
ncounts_top= 5000
ncounts_bot= 100
area_top = 30000

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=ngenes_bot)
sc.pp.filter_cells(adata, min_genes=ngenes_bot)
sc.pp.filter_cells(adata, max_counts=ncounts_top)
sc.pp.filter_cells(adata, max_genes=ngenes_top)
adata=adata[adata.obs['Area']<=area_top]
print('filtering done')

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,show=False)
plt.savefig(sample_dir_write+'post_violin_plot.pdf', format='pdf', bbox_inches='tight', dpi=300)
plt.close()

# norm+log1p 
adata.layers['raw']=adata.X.toarray()
sc.pp.normalize_total(adata, inplace=True)
count = pd.DataFrame(adata.X.toarray())
count.columns = adata.var.index.tolist()
count.index=adata.obs.index.tolist()
count.to_csv(sample_dir_write+'norm_counts.csv')
sc.pp.log1p(adata)
print('counts saved')

#fov map
plot_spatial_fov(adata, sample, save_pdf=sample_dir_write+'fov_map.pdf')
gc.collect()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist, squareform

def assign_tissue_sections(adata, fov_column='fov', spatial_key='spatial_fov', 
                           method='dbscan', eps=None, min_samples=1, 
                           distance_percentile=5):
    """
    Automatically assign tissue sections based on spatial proximity of FOVs.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object with spatial coordinates
    fov_column : str
        Column name in adata.obs containing FOV labels
    spatial_key : str
        Key in adata.obsm containing spatial coordinates
    method : str
        Clustering method: 'dbscan' or 'hierarchical'
    eps : float or None
        Maximum distance between FOVs in same tissue (DBSCAN). 
        If None, automatically calculated from distance distribution
    min_samples : int
        Minimum number of FOVs to form a tissue section
    distance_percentile : float
        Percentile of distances to use for automatic eps calculation (default: 10)
    
    Returns
    -------
    adata : AnnData
        Modified adata with 'tissue_section' column added to adata.obs
    """
    
    # Get spatial coordinates
    coords = adata.obsm[spatial_key]
    fovs = adata.obs[fov_column].unique()
    
    # Calculate FOV centroids
    fov_centroids = []
    fov_labels = []
    
    for fov in fovs:
        fov_mask = adata.obs[fov_column] == fov
        fov_coords = coords[fov_mask]
        centroid = fov_coords.mean(axis=0)
        fov_centroids.append(centroid)
        fov_labels.append(fov)
    
    fov_centroids = np.array(fov_centroids)
    
    # Auto-calculate eps if not provided
    if eps is None and method == 'dbscan':
        # Calculate pairwise distances between FOV centroids
        distances = pdist(fov_centroids)
        # Use a percentile of distances as threshold
        eps = np.percentile(distances, distance_percentile)
        print(f"Auto-calculated eps (distance threshold): {eps:.2f}")
        print(f"Distance range: {distances.min():.2f} - {distances.max():.2f}")
    
    # Perform clustering
    if method == 'dbscan':
        clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(fov_centroids)
        labels = clustering.labels_
        
    elif method == 'hierarchical':
        from scipy.cluster.hierarchy import fcluster, linkage
        if eps is None:
            distances = pdist(fov_centroids)
            eps = np.percentile(distances, distance_percentile)
        Z = linkage(fov_centroids, method='ward')
        labels = fcluster(Z, t=eps, criterion='distance') - 1
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Create mapping from FOV to tissue section
    fov_to_tissue = dict(zip(fov_labels, labels))
    
    # Handle noise points (label = -1 in DBSCAN) as separate tissues
    if -1 in labels:
        noise_fovs = [fov for fov, label in fov_to_tissue.items() if label == -1]
        max_label = max([l for l in labels if l != -1]) if any(l != -1 for l in labels) else -1
        for i, fov in enumerate(noise_fovs):
            fov_to_tissue[fov] = max_label + 1 + i
    
    # Assign tissue sections to all cells
    adata.obs['tissue_section'] = adata.obs[fov_column].map(fov_to_tissue)
    adata.obs['tissue_section'] = 'Tissue_' + adata.obs['tissue_section'].astype(str)
    
    # Print summary
    n_tissues = len(adata.obs['tissue_section'].unique())
    print(f"\nFound {n_tissues} tissue sections:")
    tissue_summary = adata.obs.groupby('tissue_section')[fov_column].nunique().sort_values(ascending=False)
    for tissue, n_fovs in tissue_summary.items():
        print(f"  {tissue}: {n_fovs} FOVs")
    
    return adata


def visualize_tissue_sections(adata, spatial_key='spatial_fov', 
                               tissue_column='tissue_section', 
                               fov_column='fov', save_pdf=None):
    """
    Visualize the assigned tissue sections with FOV boundaries.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    coords = adata.obsm[spatial_key]
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))
    
    # Plot 1: Color by tissue section
    tissues = adata.obs[tissue_column].unique()
    n_tissues = len(tissues)
    palette = sns.color_palette("tab10" if n_tissues <= 10 else "tab20", n_tissues)
    tissue_to_color = dict(zip(sorted(tissues), palette))
    
    colors = [tissue_to_color[t] for t in adata.obs[tissue_column]]
    ax1.scatter(coords[:, 0], coords[:, 1], c=colors, s=1, alpha=0.6)
    ax1.set_title('Tissue Sections', fontsize=14)
    ax1.set_xlabel('X coordinate')
    ax1.set_ylabel('Y coordinate')
    
    # Add legend
    handles = [plt.Line2D([0], [0], marker='o', color='w',
                         markerfacecolor=tissue_to_color[t], markersize=10, linestyle='None')
              for t in sorted(tissues)]
    ax1.legend(handles, sorted(tissues), title='Tissue Section', 
              loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
    
    # Plot 2: Show FOV centroids with tissue assignment
    fovs = adata.obs[fov_column].unique()
    fov_info = []
    
    for fov in fovs:
        fov_mask = adata.obs[fov_column] == fov
        fov_coords = coords[fov_mask]
        centroid = fov_coords.mean(axis=0)
        tissue = adata.obs.loc[fov_mask, tissue_column].iloc[0]
        fov_info.append({'fov': fov, 'x': centroid[0], 'y': centroid[1], 'tissue': tissue})
    
    fov_df = pd.DataFrame(fov_info)
    
    # Plot cells lightly
    ax2.scatter(coords[:, 0], coords[:, 1], c='lightgray', s=0.5, alpha=0.3)
    
    # Plot FOV centroids colored by tissue
    for tissue in sorted(tissues):
        tissue_fovs = fov_df[fov_df['tissue'] == tissue]
        ax2.scatter(tissue_fovs['x'], tissue_fovs['y'], 
                   c=[tissue_to_color[tissue]], s=200, 
                   edgecolors='black', linewidths=2,
                   label=tissue, alpha=0.8)
        
        # Add FOV labels
        for _, row in tissue_fovs.iterrows():
            ax2.annotate(row['fov'], (row['x'], row['y']), 
                        fontsize=6, ha='center', va='center')
    
    ax2.set_title('FOV Centroids by Tissue Section', fontsize=14)
    ax2.set_xlabel('X coordinate')
    ax2.set_ylabel('Y coordinate')
    ax2.legend(title='Tissue Section', loc='upper left', 
              bbox_to_anchor=(1.02, 1), frameon=False)
    
    plt.tight_layout()
    
    if save_pdf:
        plt.savefig(save_pdf, format='pdf', bbox_inches='tight', dpi=300)
        print(f"Visualization saved to: {save_pdf}")
    
    plt.close()
    
    return fig


In [ ]:
# Example usage:
# Automatic assignment with DBSCAN (recommended)
adata = assign_tissue_sections(adata, fov_column='fov', spatial_key='spatial_fov',distance_percentile=3)

# Visualize the results
visualize_tissue_sections(adata, save_pdf=sample_dir_write+'tissue_sections.pdf')

# If you want more control over the distance threshold:
# adata = assign_tissue_sections(adata, eps=5000, distance_percentile=15)

# Or use hierarchical clustering:
#adata = assign_tissue_sections(adata, method='hierarchical', distance_percentile=30)

In [ ]:
adata.write_h5ad(sample_dir_write+'norm_log1p.h5ad')
print('h5ad saved')